In [1]:
!pip install quimb mps-to-circuit qiskit qiskit_aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 10.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 24.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 44.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.9/952.9 kB 29.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 8.6 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [mps-to-circuit]m [qiskit_aer]


In [1]:
"""
main.py -- first-order Trotter sweep of the ground-state filter on a grid of n
equal steps (T = sum t_i = n * dt, every pulse t_i = k_i * dt).

Noiseless by design: exact statevector / exact-unitary propagation throughout,
so this isolates algorithmic (Trotter + filter-design + spectral-input) error
rather than device error. See builder.py's module docstring.

Audit items handled here: A1 (phase alignment), A2 (minimax + certified
selection), A3 (certified window, gap mode), A4 (rigorous bound, unclamped,
violation is fatal, extended n), A6/A7 (ordering + MPO==Pauli checks),
A8 (gamma reported), A9 (DMRG once), A11 (trial + connectivity + T-count),
A12 (sparse exact reference), A13 (home-module imports), A14 (rename),
A15 (J2 and N sweeps).
"""
import csv

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from qiskit.quantum_info import Statevector
from qiskit.transpiler import CouplingMap

from builder import (
    FilterBuilder, apply_filter_pulse, build_early_abort_circuit,
    certify_filter, cx_depth_per_step, fidelity_lower_bound, get_spectrum,
    grid_filter, lie_trotter_order, overlaps, postselected_run,
    postselected_run_exact, resource_costs, select_best, state_metrics,
    trotter_alpha, trotter_bounds, verify_pulse_convention,
)
from get_energies import get_energies
from hamiltonians import (
    MPO_ham_j1j2, check_mpo_matches_pauli, j1j2_hamiltonian, scale_hamiltonian,
)
from mps_to_circuit import mps_to_circuit

# ----------------------------------------------------------------------
# Settings
# ----------------------------------------------------------------------
SIZES = [8]                                    # extend, e.g. [6, 8, 10, 12]
J1 = 1.0
J2_LIST = [0.0, 0.2411, 0.5, 0.7, 1.0]         # free / critical / MG / 0.7 / 1.0
N_SWEEP = [10, 20, 40, 80, 160, 320, 640, 1280, 2560]
SWEEP_L = 1                                    # trial-circuit layers
VERIFY_COST_NS = (10, 160)                     # full transpile (slow at large n)
ORDER = 1                                      # first-order Trotter
SEED = 1234
BANDWIDTH = "certified"                        # or "dmrg" (uncertified)
GAP_MODE = "any"                               # or "sector"
MPO_CHECK_MAX_N = 10
COST_TOPOLOGY = None                           # None (all-to-all) or "line"
SYNTH_EPS = 1e-3                               # per-rotation synthesis precision

# ----------------------------------------------------------------------
# 0. Global convention checks -- everything below depends on these
# ----------------------------------------------------------------------
if not verify_pulse_convention():
    raise RuntimeError("verify_pulse_convention() failed; fix before trusting "
                       "any fidelity or P_succ number.")
_Htest = j1j2_hamiltonian(4, 1.0, 0.7)
_ord = lie_trotter_order(_Htest)
print(f"LieTrotter term order: {_ord}")
if not (_ord["forward"] or _ord["reverse"]):
    raise RuntimeError("LieTrotter does not equal either explicit ordered "
                       "product; the commutator bound's ordering is unknown.")


def run_case(N, j1, j2):
    tag = f"N{N}_J2_{j2:g}"
    print(f"\n{'=' * 70}\nCASE {tag}  (J1={j1}, J2={j2})\n{'=' * 70}")

    # --- A7: MPO (DMRG input) == Pauli form (circuit) -------------------
    if N <= MPO_CHECK_MAX_N:
        err = check_mpo_matches_pauli(N, j1, j2)
        print(f"MPO vs Pauli Hamiltonian: max |dH| = {err:.2e}")

    # --- A9: DMRG exactly once ----------------------------------------
    H_mpo = MPO_ham_j1j2(N, j1=j1, j2=j2, cyclic=False)
    res = get_energies(H_mpo, N=N, j1=j1, j2=j2, run_ed_check=True)
    E0_dmrg, psi0_mps, ed = float(np.real(res["E0"])), res["psi0"], res["ed"]
    psi0_ed = ed.get("psi0_ed") if ed else None      # MPS ordering
    psi0_dmrg = np.asarray(psi0_mps.to_dense()).reshape(-1)
    psi0_dmrg = psi0_dmrg / np.linalg.norm(psi0_dmrg)

    if psi0_ed is not None:
        dmrg_ed = float(abs(np.vdot(psi0_ed, psi0_dmrg)) ** 2)
        print(f"E0 DMRG = {E0_dmrg:.8f}   |<DMRG|ED>|^2 = {dmrg_ed:.8f}")
        if dmrg_ed < 1 - 1e-4:
            print(f"  [note] F_dmrg inherits up to ~{1 - dmrg_ed:.1e} of DMRG's "
                  f"own error.")
    else:
        dmrg_ed = float("nan")
        print(f"E0 DMRG = {E0_dmrg:.8f}   (no ED reference at N={N})")

    # --- spectrum, certified window, rescaled Hamiltonian ---------------
    H_qk = j1j2_hamiltonian(N, j1=j1, j2=j2)
    spec = get_spectrum(N, j1=j1, j2=j2, source="dmrg", res=res,
                        bandwidth=BANDWIDTH, gap_mode=GAP_MODE)
    H_scaled_qk = scale_hamiltonian(H_qk, spec["shift"], spec["W"])
    Hs = H_scaled_qk.to_matrix(sparse=True)
    if N <= 12:
        w = np.linalg.eigvalsh(Hs.toarray())
        print(f"scaled spectrum in [{w[0]:.3e}, {w[-1]:.6f}]")
        if w[0] < -1e-6 or w[-1] > 1 + 1e-9:
            print("  [warning] scaled spectrum leaves [0,1]; the certification "
                  "assumption (spectrum in {E0} U [Delta,1]) is violated.")

    # --- optimize filter (minimax, multistart), select by certified eta -
    builder = FilterBuilder(total_time=spec["totaltime"],
                            energies=spec["energies"], seed=SEED,
                            e0_slack=spec["e0_slack"])
    best = select_best(builder.build())
    sw_times, sw_phases = best["times"], best["phases"]
    T = float(np.sum(sw_times))
    print(f"\nSelected: {len(sw_times)} pulses, eta_cert={best['eta']:.3e}, "
          f"ok={best['ok']}, T={T:.4f} (pi/gap={spec['totaltime']:.4f}), "
          f"t_i in [{sw_times.min():.3f}, {sw_times.max():.3f}]")
    for t in sorted(sw_times):
        print(f"  {t:.4f}")

    # --- trial circuit and overlaps gamma (A8) ---------------------------
    trial_qc = mps_to_circuit(psi0_mps.arrays, method="approximate",
                              shape="lpr", num_layers=SWEEP_L)
    if trial_qc.num_qubits != N:
        raise ValueError(f"trial circuit has {trial_qc.num_qubits} qubits, "
                         f"expected {N}")
    g_dmrg, g_ed = overlaps(Statevector(trial_qc).data, psi0_dmrg, psi0_ed)
    gamma_ref = g_ed if np.isfinite(g_ed) else g_dmrg
    F_lb = fidelity_lower_bound(gamma_ref, best["eta"])
    print(f"trial overlap gamma_dmrg={g_dmrg:.6f}  gamma_ed={g_ed:.6f}  "
          f"(layer-{SWEEP_L} truncation error 1-gamma_dmrg={1 - g_dmrg:.2e})")
    print(f"certified exact-filter fidelity lower bound = {F_lb:.6f}")

    # --- exact reference with the optimizer's (ungridded) times ----------
    p_ex, sv_ex = postselected_run_exact(trial_qc, sw_times, sw_phases, Hs)
    E_ex, Fd_ex, Fe_ex = state_metrics(sv_ex, H_qk, psi0_dmrg, psi0_ed)
    print(f"[exact, optimizer times]  P_succ={p_ex:.6f}  "
          f"E-E0dmrg={E_ex - E0_dmrg:.3e}  F_dmrg={Fd_ex:.6f}  F_ed={Fe_ex:.6f}")

    # --- costs and commutator constant -----------------------------------
    cx1, d1 = cx_depth_per_step(H_scaled_qk, order=ORDER)
    alpha = trotter_alpha(H_scaled_qk, tight=True)
    print(f"alpha (max over term orders) = {alpha:.4f}   "
          f"CX/step = {cx1}   depth/step = {d1}")
    trial_cost = resource_costs(trial_qc, label=f"trial circuit L={SWEEP_L}",
                                synth_eps=SYNTH_EPS)
    cmap = CouplingMap.from_line(N + 1) if COST_TOPOLOGY == "line" else None

    # --- sweep over n ----------------------------------------------------
    # bound_state = 2*eps/sqrt(p_g) is now a THEOREM (builder.trotter_bounds):
    # a violation means a bug (term order, convention, alpha), so it is fatal.
    rows = []
    for n in N_SWEEP:
        k, dt, ph = grid_filter(sw_times, sw_phases, T, n, spec["energies"])
        tg = k * dt
        cert_g = certify_filter(tg, ph, spec["energies"], e0_slack=spec["e0_slack"])

        p_g, sv_g = postselected_run_exact(trial_qc, tg, ph, Hs)
        _, Fd_g, Fe_g = state_metrics(sv_g, H_qk, psi0_dmrg, psi0_ed)

        p, sv = postselected_run(
            trial_qc, tg, ph,
            lambda qc, s, a, t, phi, i, k=k: apply_filter_pulse(
                qc, s, a, H_scaled_qk, t, phi, int(k[i]), order=ORDER))
        E, Fd, Fe = state_metrics(sv, H_qk, psi0_dmrg, psi0_ed)
        F_vs_grid = float(abs(np.vdot(sv_g, sv)) ** 2)

        bnd = trotter_bounds(alpha, tg, k, p_g)
        # A1: phase-align with exp(-i theta) so that <sv_g | e^{-i theta} sv> > 0
        theta = np.angle(np.vdot(sv_g, sv))
        state_err = float(np.linalg.norm(sv_g - np.exp(-1j * theta) * sv))

        if state_err > bnd["bound_state"] + 1e-9:
            raise RuntimeError(
                f"BOUND VIOLATION at n={n}: state_err={state_err:.3e} > "
                f"{bnd['bound_state']:.3e}. Check term order, alpha, convention.")
        F_lb_g = fidelity_lower_bound(gamma_ref, cert_g["eta"])
        F_exact_ref = Fe_g if np.isfinite(Fe_g) else Fd_g
        if F_exact_ref < F_lb_g - 1e-6:
            print(f"  [warning] n={n}: exact-grid fidelity {F_exact_ref:.6f} < "
                  f"certified lower bound {F_lb_g:.6f} (spectral assumption "
                  f"or gamma reference is off).")

        row = {
            "N": N, "J1": j1, "J2": j2, "n": n, "dt": dt, "n_pulses": len(k),
            "k": k.tolist(), "alpha": alpha,
            "bound_total": bnd["bound_total"],
            "bound_state_raw": bnd["bound_state_raw"],
            "bound_state": bnd["bound_state"], "state_err": state_err,
            "eta_grid": cert_g["eta"], "F_lb_grid": F_lb_g,
            "gamma_dmrg": g_dmrg, "gamma_ed": g_ed,
            "P_succ_exact_grid": p_g, "P_succ": p,
            "F_dmrg_exact_grid": Fd_g, "F_ed_exact_grid": Fe_g,
            "F_dmrg": Fd, "F_ed": Fe, "E-E0_dmrg": E - E0_dmrg,
            "F_vs_exact_grid": F_vs_grid,
            "cx": cx1 * n, "depth": d1 * n,
        }
        rows.append(row)
        print(f"[n={n:>5}] dt={dt:.4f} k={k.tolist()}  P_succ={p:.6f} "
              f"(grid exact {p_g:.6f})  F_dmrg={Fd:.6f} (grid exact {Fd_g:.6f})  "
              f"err={state_err:.2e}  bound={bnd['bound_state_raw']:.2e}  "
              f"eta_grid={cert_g['eta']:.2e}  CX~{row['cx']}")

        if n in VERIFY_COST_NS:
            qc_r, _ = build_early_abort_circuit(
                H_scaled_qk, tg, ph, trial_prep=None,
                trotter_steps=[int(x) for x in k], order=ORDER)
            c = resource_costs(qc_r, label=f"filter n={n} (full transpile)",
                               flatten=True, coupling_map=cmap,
                               synth_eps=SYNTH_EPS)
            est_cx = row["cx"]
            pct = 100 * (c["cx"] - est_cx) / est_cx if est_cx else float("nan")
            print(f"  cost check n={n}: measured CX={c['cx']} depth={c['depth']} "
                  f"vs estimate CX={est_cx} depth={row['depth']} "
                  f"(CX off by {pct:+.1f}%)")
            print(f"  total incl. trial: CX={c['cx'] + trial_cost['cx']}  "
                  f"non-Clifford rz={c['rz_nonclifford'] + trial_cost['rz_nonclifford']}"
                  f"  approx T={c['t_count_est'] + trial_cost['t_count_est']:.0f}")

    vac = [r["n"] for r in rows if r["bound_state_raw"] < 2.0]
    print(f"bound non-vacuous (<2) from n = {vac[0] if vac else 'beyond the sweep'}")

    # --- save ------------------------------------------------------------
    with open(f"trotter_sweep_{tag}.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)

    # --- plots -----------------------------------------------------------
    ns = [r["n"] for r in rows]
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    ax[0].loglog(ns, [max(r["state_err"], 1e-16) for r in rows], "o-",
                 label="actual (phase-aligned vector norm)")
    ax[0].loglog(ns, [r["bound_state_raw"] for r in rows], "k--",
                 label=r"bound $2\alpha T^2/(2n\sqrt{p_g})$ (unclamped)")
    ax[0].axhline(2.0, color="gray", ls=":", label="trivial bound 2")
    ax[0].set_xlabel("total Trotter steps n (dt = T/n)")
    ax[0].set_ylabel("state error vs exact grid filter")
    ax[0].set_title("Trotter error vs rigorous bound")
    ax[0].legend()

    ax[1].semilogx(ns, [r["F_dmrg"] for r in rows], "o-", color="C0",
                   label="F vs DMRG (Trotter)")
    ax[1].semilogx(ns, [r["F_dmrg_exact_grid"] for r in rows], "o:", color="C0",
                   alpha=0.6, label="F vs DMRG (exact, grid)")
    ax[1].set_xlabel("total Trotter steps n")
    ax[1].set_ylabel("post-filter fidelity", color="C0")
    axr = ax[1].twinx()
    axr.semilogx(ns, [r["P_succ"] for r in rows], "s-", color="C1",
                 label="P_succ (Trotter)")
    axr.semilogx(ns, [r["P_succ_exact_grid"] for r in rows], "s:", color="C1",
                 alpha=0.6, label="P_succ (exact, grid)")
    axr.set_ylabel("success probability", color="C1")
    ax[1].set_title("Fidelity and success prob (dotted = exact)")
    l1, n1 = ax[1].get_legend_handles_labels()
    l2, n2 = axr.get_legend_handles_labels()
    ax[1].legend(l1 + l2, n1 + n2, loc="lower right")

    ax[2].loglog(ns, [r["cx"] for r in rows], "o-", label="CX (estimate)")
    ax[2].loglog(ns, [r["depth"] for r in rows], "s-", label="depth (estimate)")
    ax[2].set_xlabel("total Trotter steps n")
    ax[2].set_ylabel("count (basis h,s,cx,rz)")
    ax[2].set_title("Filter cost (estimate; see cost-check prints)")
    ax[2].legend()

    fig.suptitle(f"First-order Trotter, N={N}, L={SWEEP_L}, J2/J1={j2 / j1:g}, "
                 f"bandwidth={BANDWIDTH}, gap={GAP_MODE}")
    fig.tight_layout()
    fig.savefig(f"trotter_sweep_{tag}.png", dpi=150)
    plt.close(fig)
    print(f"saved trotter_sweep_{tag}.csv/.png")

    last = rows[-1]
    return dict(N=N, J1=j1, J2=j2, gap_scaled=spec["gap"], W=spec["W"],
                T=T, n_pulses=len(sw_times), eta_cert=best["eta"],
                gamma_dmrg=g_dmrg, gamma_ed=g_ed, F_lb_exact=F_lb,
                dmrg_ed_overlap=dmrg_ed, F_dmrg_exact_ref=Fd_ex,
                F_dmrg_n_max=last["F_dmrg"], P_succ_n_max=last["P_succ"],
                n_nonvacuous=vac[0] if vac else -1)


if __name__ == "__main__":
    summary = []
    for N_ in SIZES:
        for J2_ in J2_LIST:
            summary.append(run_case(N_, J1, J2_))
    with open("case_summary.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(summary[0].keys()))
        w.writeheader()
        w.writerows(summary)
    print("saved case_summary.csv")

/opt/conda/lib/python3.12/site-packages/scipy/sparse/linalg/_dsolve/linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/opt/conda/lib/python3.12/site-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)


pulse convention: circuit err=8.54e-16  numpy err=4.35e-16  classical err=2.01e-16  flipped-phi err=7.23e-01 (must be large)  ->  OK
reps=1: forward=1.046e-14, reverse=1.972e-03
reps=3: forward=7.939e-15, reverse=5.914e-03
LieTrotter term order: {'forward': True, 'reverse': False, 'err_forward': 1.045876143692981e-14, 'err_reverse': 0.005913519447381724}

CASE N8_J2_0  (J1=1.0, J2=0.0)
MPO vs Pauli Hamiltonian: max |dH| = 1.34e-15
Running DMRG for ground state...
Running DMRG for highest excited state (via -H)...
  spectral bandwidth = 5.12493-2.22045e-16j, PENALTY = 5.63743-2.44249e-16j
Running DMRG for first excited state...
Running ED cross-check (N=8, dim=2^8=256)...
  [OK] E0: DMRG=-3.3749325987-0.0000000000j  ED=-3.3749325987  |diff|=4.58e-16
  [OK] E1: DMRG=-2.9822404878  ED=-2.9822404878  |diff|=2.22e-15
  [OK] E_top: DMRG=1.7500000000-0.0000000000j  ED=1.7500000000  |diff|=2.47e-15
E0 DMRG = -3.37493260   |<DMRG|ED>|^2 = 1.00000000
[spectrum] E0=-3.37493260 E1=-2.98224049 Etop

In [5]:
#debug cell

from typing import Dict, List, Optional
from builder import *
import matplotlib.pyplot as plt
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as sla
from scipy import optimize as opt
from scipy.linalg import expm
from scipy.sparse.linalg import expm_multiply

from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister, transpile
from qiskit.circuit import ControlFlowOp, IfElseOp
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.quantum_info import Operator, SparsePauliOp, Statevector
from qiskit.synthesis import LieTrotter, MatrixExponential, SuzukiTrotter
def debug_lie(H, tau=0.05, reps=1):
    N = H.num_qubits
    t = tau * reps

    # Manual decomposition
    mats = [
        SparsePauliOp(H.paulis[i], [H.coeffs[i]]).to_matrix()
        for i in range(len(H))
    ]

    dt = t / reps

    Uf = np.eye(2**N, dtype=complex)
    Ur = np.eye(2**N, dtype=complex)

    for m in mats:
        s = expm(-1j * dt * m)
        Uf = s @ Uf
        Ur = Ur @ s

    Uf = np.linalg.matrix_power(Uf, reps)
    Ur = np.linalg.matrix_power(Ur, reps)

    # Qiskit
    qc = QuantumCircuit(N)
    qc.append(evolution_gate(H, t, reps, 1), range(N))
    U = Operator(qc).data

    print("forward:", np.linalg.norm(U - Uf, 2))
    print("reverse:", np.linalg.norm(U - Ur, 2))

    # Exact evolution
    Hmat = H.to_matrix()
    Uexact = expm(-1j * t * Hmat)

    print("exact:", np.linalg.norm(U - Uexact, 2))

    return U, Uf, Ur

U, Uf, Ur = debug_lie(j1j2_hamiltonian(4, 1.0, 0.7), reps=1)

forward: 0.0009861512580814082
reverse: 0.0009861512580814065
exact: 2.2753090467284455e-16
